# Integration (10 degree mosaicity)

Every detector frame is integrated azimuthally over a q-window around each of
the selected powder rings, giving the array `I[translation, omega, eta, ring]` that the
texture tomography is fitted to. The array is written as a raw memmap (`.bin`)
next to a `.json` with all parameters (see `integration/integrated_data.py`).

The dataset-specific reading lives in `integration/frame_loader.py`. This
notebook only uses its generic interface.

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import sys
from pathlib import Path

# make the repository importable, whatever the working directory is
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "simtools").is_dir())
sys.path.insert(0, str(REPO))

import json
import os
import time
from concurrent.futures import ProcessPoolExecutor, wait, FIRST_COMPLETED
from datetime import datetime, timezone
from multiprocessing import shared_memory

import numpy as np
import matplotlib.pyplot as plt
import pyFAI
from matplotlib.patches import Rectangle
from scipy.ndimage import maximum_filter
from diffractom import Material
from integration.frame_loader import SimulatedDataset, PONI_PATH, PARAMETERS_AL, read_batch_into_shm, dark
from integration.integrated_data import IntegratedData
from simtools import paths

dark()

In [ ]:
SAMPLE = "domains_mosaicity_10p0deg"
INTEGRATED_PATH = paths.integrated_path(SAMPLE)

## A few frames
Frames are max-filtered for display only, so that peaks of a few pixels stay visible.

In [ ]:
target_trans_frac = [0.25, 0.5, 0.75]
target_omega_deg = [0, 45, 90, 135, 179.5]

with SimulatedDataset(SAMPLE) as ds:
    n_trans = ds.n_translations
    i_trans_list = [int(round(f * (n_trans - 1))) for f in target_trans_frac]
    fig, axes = plt.subplots(len(i_trans_list), len(target_omega_deg),
                             figsize=(3.2 * len(target_omega_deg), 3.2 * len(i_trans_list)), squeeze=False)
    for row, i_trans in enumerate(i_trans_list):
        omega = ds.omega(i_trans)
        dty_val = ds.translations()[i_trans]
        for col, target in enumerate(target_omega_deg):
            i_omega = int(np.argmin(np.abs(omega - target)))
            frame = ds.frame(i_trans, i_omega)
            ax = axes[row, col]
            im = ax.imshow(maximum_filter(frame, size=15), vmin=0.1, vmax=50, norm="log", cmap="jet")
            ax.set_title(f"dty={dty_val:.2f}\nomega={omega[i_omega]:.1f}", fontsize=9)
            ax.axis("off")
    fig.colorbar(im, ax=axes, fraction=0.02, pad=0.02, label="Intensity")
    plt.show()

## Choose the rings
The frames above are integrated with a fine radial binning, and the result is compared with the powder peaks predicted from the unit cell in `Al.par`.

In [ ]:
ai = pyFAI.load(PONI_PATH)
npt_rad = 2000
npt_azim = 180

def read_par_file(path):
    params = {}
    with open(path) as f:
        for line in f:
            parts = line.split()
            if len(parts) >= 2:
                params[parts[0]] = parts[1]
    return params

I_sum, q, chi = None, None, None
with SimulatedDataset(SAMPLE) as ds:
    mask = ds.mask()
    for i_trans in i_trans_list:
        omega = ds.omega(i_trans)
        for target in target_omega_deg:
            i_omega = int(np.argmin(np.abs(omega - target)))
            res = ai.integrate2d(ds.frame(i_trans, i_omega), npt_rad=npt_rad, npt_azim=npt_azim,
                                 unit="q_nm^-1", mask=mask, method=("no", "csr", "opencl"))
            if I_sum is None:
                I_sum = np.zeros_like(res.intensity)
                q, chi = res.radial, res.azimuthal
            I_sum += res.intensity
I_sum = np.nan_to_num(I_sum)

par = read_par_file(PARAMETERS_AL)
mat = Material.from_lattice_parameters(
    a=float(par["cell__a"]), b=float(par["cell__b"]), c=float(par["cell__c"]),
    alpha=float(par["cell_alpha"]), beta=float(par["cell_beta"]), gamma=float(par["cell_gamma"]),
    space_group_number=int(par["cell_lattice_[P,A,B,C,I,F,R]"]),
    wavelength_A=float(par["wavelength"]),
    q_min=q.min() / 10, q_max=q.max() / 10,  # nm^-1 -> A^-1
)
q_peaks_nm = np.unique(np.round(2 * np.pi / mat.reflections["d_spacing"] * 10, 8))
print(f"{len(q_peaks_nm)} distinct predicted peaks within the detector's q-range")

The 8 lowest-q rings (111, 200, 220, 311, 222, 400, 331, 420) are used,
each integrated over a window of full width `Q_WIDTH_NM` = 2 nm^-1 (±0.1 1/A).
These are the rings and window used in the article.

In [ ]:
Q_WIDTH_NM = 2.0  # full width of the window around each ring, nm^-1
N_RINGS = 8

selected_peaks = q_peaks_nm[:N_RINGS]
excluded_peaks = q_peaks_nm[N_RINGS:]

fig, ax = plt.subplots(1, 1, figsize=(14, 10))
vmin = max(I_sum[I_sum > 0].min(), 1e-3)
im = ax.imshow(I_sum, extent=[q.min(), q.max(), chi.min(), chi.max()], origin="lower", aspect="auto",
               vmin=vmin, vmax=np.percentile(I_sum, 99.99), norm="log", cmap="jet")
for peak in selected_peaks:
    ax.add_patch(Rectangle((peak - Q_WIDTH_NM / 2, chi.min()), Q_WIDTH_NM, chi.max() - chi.min(),
                           facecolor="red", alpha=0.3, edgecolor=None))
for peak in excluded_peaks:
    ax.vlines(peak, chi.min(), chi.max(), color="red", linewidth=1.5, linestyle="--", alpha=0.55)
ax.set_xlabel(r"q (nm$^{-1}$)")
ax.set_ylabel("azimuthal angle (deg)")
fig.colorbar(im, ax=ax, fraction=0.052, pad=0.04, label="summed intensity")
plt.tight_layout()
plt.show()

profile = I_sum.sum(axis=0)
fig, ax = plt.subplots(1, 1, figsize=(14, 5))
ax.semilogy(q, profile + 1e-3)
for peak in selected_peaks:
    ax.axvspan(peak - Q_WIDTH_NM / 2, peak + Q_WIDTH_NM / 2, color="red", alpha=0.3)
ax.set_xlabel(r"q (nm$^{-1}$)")
ax.set_ylabel("summed intensity")
plt.show()

## Parameters
The azimuthal axis: `N_ETA` bins of 2 deg over pyFAI's (-180, 180) range. Everything is written to the json before the integration starts.

In [ ]:
N_ETA = 180
ETA_RANGE_DEG = (-180.0, 180.0)
eta_edges = np.linspace(ETA_RANGE_DEG[0], ETA_RANGE_DEG[1], N_ETA + 1)
eta_centres = 0.5 * (eta_edges[:-1] + eta_edges[1:])

json_path = INTEGRATED_PATH + ".json"
bin_path = INTEGRATED_PATH + ".bin"

with SimulatedDataset(SAMPLE) as ds:
    translations = ds.translations()
    n_trans = ds.n_translations
    omega = ds.omega(0)  # rotation sequence, identical for every translation
    n_omega = len(omega)
    for i_trans in range(n_trans):
        assert np.allclose(ds.omega(i_trans), omega)

def reflections_for_q(mat, q_values_nm, tol=1e-6):
    """Match each |q| ring back to every (h,k,l) family that lands on it."""
    q_all_nm = 2 * np.pi / mat.reflections["d_spacing"] * 10
    out = []
    for qv in q_values_nm:
        matches = np.where(np.abs(q_all_nm - qv) < tol)[0]
        out.append({
            "q_nm": float(qv),
            "hkl": [mat.reflections["hkl"][i].tolist() for i in matches],
            "multiplicity": [int(mat.reflections["multiplicity"][i]) for i in matches],
            "d_spacing_A": float(mat.reflections["d_spacing"][matches[0]]),
            "two_theta_rad": float(mat.reflections["two_theta"][matches[0]]),
        })
    return out

params = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "integration_complete": False,
    "geometry": {
        "poni_path": PONI_PATH,
        "parameters_path": PARAMETERS_AL,
        "mask_path": None,
        "wavelength_A": ai.wavelength * 1e10,
    },
    "material": {
        "space_group_number": mat.space_group_number,
        "space_group_symbol": mat.space_group_symbol,
        "lattice_params": mat.lattice_params,
    },
    "rings": {
        "q_width_nm": Q_WIDTH_NM,
        "n_eta": N_ETA,
        "eta_range_deg": list(ETA_RANGE_DEG),
        "eta_deg": eta_centres.tolist(),
        "selected": reflections_for_q(mat, selected_peaks),
    },
    "scan": {
        "n_translations": n_trans,
        "n_omega": n_omega,
        "translations_dty": translations.tolist(),
        "omega_deg": omega.tolist(),
    },
    "output": {
        "bin_path": bin_path,
        "dtype": "float32",
        "shape": [n_trans, n_omega, N_ETA, len(selected_peaks)],
        "shape_axes": ["translation", "omega", "eta", "q_ring"],
        "memmap_order": "C",
    },
}
with open(json_path, "w") as f:
    json.dump(params, f, indent=2)
print(f"Output array shape: {params['output']['shape']}, {4 * np.prod(params['output']['shape']) / 1e9:.2f} GB")
print(f"Wrote parameters to {json_path}")

## Integrate
Worker processes read and densify batches of frames into shared memory, and the main process integrates them on the GPU. One `integrate2d` call per frame covers all rings; the rings are then summed out with a 0/1 matrix.

In [ ]:
with open(json_path) as f:
    params = json.load(f)

ai = pyFAI.load(params["geometry"]["poni_path"])
Q_WIDTH_NM = params["rings"]["q_width_nm"]
N_eta = params["rings"]["n_eta"]
eta_range = tuple(params["rings"]["eta_range_deg"])
selected_q = np.array([r["q_nm"] for r in params["rings"]["selected"]])

radial_ranges = [(q0 - Q_WIDTH_NM / 2, q0 + Q_WIDTH_NM / 2) for q0 in selected_q]
q_range_min = min(lo for lo, hi in radial_ranges)
q_range_max = max(hi for lo, hi in radial_ranges)
n_bins_per_ring = 10
npt_rad = int((q_range_max - q_range_min) / Q_WIDTH_NM * n_bins_per_ring)
N_rings = len(radial_ranges)

shape = tuple(params["output"]["shape"])
I_out = np.memmap(params["output"]["bin_path"], dtype=params["output"]["dtype"], mode="w+", shape=shape)

read_batch_size = 10
n_worker_processes = min(16, len(os.sched_getaffinity(0)))
n_shm_buffers = n_worker_processes + 2

with SimulatedDataset(SAMPLE) as ds:
    mask = ds.mask()
    frame_shape = ds.frame_shape
    dtype_str = ds.frame_dtype
    buf_nbytes = read_batch_size * np.prod(frame_shape) * np.dtype(dtype_str).itemsize
    print(f"shared-memory pool: {n_shm_buffers} buffers x {buf_nbytes/1e6:.0f} MB = {n_shm_buffers * buf_nbytes / 1e9:.2f} GB")

    res0 = ai.integrate2d(ds.frame(0, 0), npt_rad=npt_rad, npt_azim=N_eta, radial_range=(q_range_min, q_range_max),
                          azimuth_range=eta_range, unit="q_nm^-1", mask=mask, method=("no", "csr", "opencl"))
    assert np.allclose(res0.azimuthal, params["rings"]["eta_deg"], atol=1e-3), (
        "pyFAI's azimuthal bin centres don't match eta_deg in the json")
    ring_matrix = np.zeros((npt_rad, N_rings), dtype=np.float32)
    for i_ring, (lo, hi) in enumerate(radial_ranges):
        ring_matrix[(res0.radial >= lo) & (res0.radial < hi), i_ring] = 1.0

    shm_blocks = [shared_memory.SharedMemory(create=True, size=int(buf_nbytes)) for _ in range(n_shm_buffers)]
    shm_arrays = {shm.name: np.ndarray((read_batch_size,) + frame_shape, dtype=dtype_str, buffer=shm.buf)
                  for shm in shm_blocks}

    def integrate_translation(pool, path, abs_start, abs_stop, out_view, free_names):
        starts = list(range(abs_start, abs_stop, read_batch_size))
        pending = {}
        next_idx = 0

        def submit_next():
            nonlocal next_idx
            if next_idx >= len(starts) or not free_names:
                return
            name = free_names.pop()
            s = starts[next_idx]
            stop = min(s + read_batch_size, abs_stop)
            fut = pool.submit(read_batch_into_shm, path, s, stop, name, read_batch_size, frame_shape, dtype_str)
            pending[fut] = (s, stop, name)
            next_idx += 1

        while free_names and next_idx < len(starts):
            submit_next()

        while pending:
            done, _ = wait(pending.keys(), return_when=FIRST_COMPLETED)
            for fut in done:
                s, stop, name = pending.pop(fut)
                n = fut.result()
                buf = shm_arrays[name]
                batch_out = np.empty((n, N_eta, N_rings), dtype=np.float32)
                for i in range(n):
                    res = ai.integrate2d(buf[i], npt_rad=npt_rad, npt_azim=N_eta, radial_range=(q_range_min, q_range_max),
                                         azimuth_range=eta_range, unit="q_nm^-1", mask=mask, method=("no", "csr", "opencl"))
                    batch_out[i] = res.intensity @ ring_matrix
                out_view[s - abs_start:s - abs_start + n] = batch_out
                free_names.append(name)
                submit_next()

    try:
        with ProcessPoolExecutor(max_workers=n_worker_processes) as pool:
            free_names = [shm.name for shm in shm_blocks]
            t0 = time.perf_counter()
            n_frames_done = 0
            for i_trans in range(ds.n_translations):
                assert ds.n_omega(i_trans) == shape[1]
                path, abs_start, abs_stop = ds.raw_location(i_trans)
                integrate_translation(pool, path, abs_start, abs_stop, I_out[i_trans], free_names)
                n_frames_done += ds.n_omega(i_trans)
                rate = n_frames_done / (time.perf_counter() - t0)
                if (i_trans + 1) % 10 == 0 or i_trans + 1 == ds.n_translations:
                    print(f"translation {i_trans + 1}/{ds.n_translations}  ({rate:.1f} fr/s, "
                          f"~{(shape[0] * shape[1] - n_frames_done) / rate / 60:.1f} min remaining)")
    finally:
        for shm in shm_blocks:
            shm.close()
            shm.unlink()

I_out.flush()
del I_out

params["integration_complete"] = True
with open(json_path, "w") as f:
    json.dump(params, f, indent=2)
print("Integration complete.")

## Check the result
The integrated data can be loaded with `IntegratedData`. Shown below: a sinogram (intensity summed over eta) per ring.

In [ ]:
data = IntegratedData(INTEGRATED_PATH)
print(data)

sinos = np.asarray(data.I).sum(axis=2)  # (translation, omega, ring)
fig, axes = plt.subplots(2, data.n_rings // 2, figsize=(5 * data.n_rings // 2, 8))
for i, ax in enumerate(axes.flatten()):
    ax.imshow(sinos[:, :, i], aspect="auto", cmap="viridis",
              extent=[data.omega_deg[0], data.omega_deg[-1], data.translations[-1], data.translations[0]])
    ax.set_title(f"hkl {tuple(data.hkl[i][0])}")
    ax.set_xlabel("omega (deg)")
    ax.set_ylabel("dty (um)")
plt.tight_layout()
plt.show()